In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# PrunableLinear Layer

class PrunableLinear(nn.Module):

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))
        self.gate_scores = nn.Parameter(torch.zeros(out_features, in_features))

        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:

        gates = torch.sigmoid(self.gate_scores)          


        pruned_weights = self.weight * gates             


        return F.linear(x, pruned_weights, self.bias)

    def gate_values(self) -> torch.Tensor:
   
        return torch.sigmoid(self.gate_scores).detach()

    def sparsity_loss(self) -> torch.Tensor:
        
        return torch.sigmoid(self.gate_scores).mean()

    def extra_repr(self) -> str:
        return f"in_features={self.in_features}, out_features={self.out_features}"



# Network


class SelfPruningNet(nn.Module):
 
    def __init__(self):
        super().__init__()
        self.fc1 = PrunableLinear(3 * 32 * 32, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 128)
        self.fc4 = PrunableLinear(128, 10)
        self.bn1 = nn.BatchNorm1d(512)
        self.bn2 = nn.BatchNorm1d(256)
        self.bn3 = nn.BatchNorm1d(128)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)               
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = F.relu(self.bn3(self.fc3(x)))
        return self.fc4(x)                        

    def prunable_layers(self):
        
        for m in self.modules():
            if isinstance(m, PrunableLinear):
                yield m

    def sparsity_loss(self) -> torch.Tensor:
        
        layer_losses = [l.sparsity_loss() for l in self.prunable_layers()]
        return torch.stack(layer_losses).mean()

    def sparsity_level(self, threshold: float = 1e-2) -> float:
        
        all_gates = torch.cat(
            [l.gate_values().view(-1) for l in self.prunable_layers()]
        )
        pruned = (all_gates < threshold).float().sum()
        return (pruned / all_gates.numel()).item()

    def all_gate_values(self) -> np.ndarray:
      
        return torch.cat(
            [l.gate_values().view(-1) for l in self.prunable_layers()]
        ).cpu().numpy()



#  Total Loss = ClassificationLoss + λ * SparsityLoss


def compute_loss(
    logits: torch.Tensor,
    labels: torch.Tensor,
    model: SelfPruningNet,
    lam: float,
) -> tuple[torch.Tensor, torch.Tensor]:
   
    cls_loss  = F.cross_entropy(logits, labels)
    spar_loss = model.sparsity_loss()
    return cls_loss + lam * spar_loss, cls_loss


# Data Loading


def get_cifar10_loaders(batch_size: int = 128):
   
    normalize = transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std =(0.2023, 0.1994, 0.2010),
    )
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        normalize,
    ])
    test_transform = transforms.Compose([transforms.ToTensor(), normalize])

    train_set = torchvision.datasets.CIFAR10(
        root="./data", train=True,  download=True, transform=train_transform)
    test_set  = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=True, transform=test_transform)

    train_loader = DataLoader(train_set, batch_size=batch_size,
                              shuffle=True,  num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=256,
                              shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, test_loader



#  Training Loop


def train_one_epoch(
    model: SelfPruningNet,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    lam: float,
) -> tuple[float, float]:

    model.train()
    total_sum = cls_sum = 0.0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        t_loss, c_loss = compute_loss(model(imgs), labels, model, lam)
        t_loss.backward()     
        optimizer.step()
        total_sum += t_loss.item()
        cls_sum   += c_loss.item()
    n = len(loader)
    return total_sum / n, cls_sum / n


@torch.no_grad()
def evaluate(model: SelfPruningNet, loader: DataLoader) -> float:
   
    model.eval()
    correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        correct += (model(imgs).argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return correct / total


def train(
    lam: float,
    epochs: int = 30,
    lr: float = 1e-3,
    batch_size: int = 128,
) -> tuple[float, float, np.ndarray]:
    
    train_loader, test_loader = get_cifar10_loaders(batch_size)
    model = SelfPruningNet().to(DEVICE)

   
    gate_params   = [p for n, p in model.named_parameters() if "gate_scores" in n]
    weight_params = [p for n, p in model.named_parameters() if "gate_scores" not in n]
    optimizer = optim.Adam([
        {"params": weight_params, "lr": lr,       "weight_decay": 1e-4},
        {"params": gate_params,   "lr": lr * 10,  "weight_decay": 0.0},
    ])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"\n{'='*55}")
    print(f"  Training with λ = {lam}")
    print(f"{'='*55}")

    for epoch in range(1, epochs + 1):
        t_loss, c_loss = train_one_epoch(model, train_loader, optimizer, lam)
        scheduler.step()

        if epoch % 5 == 0 or epoch == 1:
            acc  = evaluate(model, test_loader)
            spar = model.sparsity_level()
            print(
                f"  Epoch {epoch:3d}/{epochs} | "
                f"Total loss: {t_loss:.4f} | CLS loss: {c_loss:.4f} | "
                f"Test acc: {acc:.3f} | Sparsity: {spar:.2%}"
            )

    final_acc  = evaluate(model, test_loader)
    final_spar = model.sparsity_level()
    gate_vals  = model.all_gate_values()
    print(f"\n  ✓ Final  →  Accuracy: {final_acc:.3f}  |  Sparsity: {final_spar:.2%}")
    return final_acc, final_spar, gate_vals



# Plotting


def plot_gate_distributions(
    results: dict,
    save_path: str = "gate_distribution.png",
):
    
    lambdas = sorted(results.keys())
    fig, axes = plt.subplots(len(lambdas), 2, figsize=(14, 4 * len(lambdas)))
    if len(lambdas) == 1:
        axes = [axes]

    for i, lam in enumerate(lambdas):
        acc, spar, gates = results[lam]
        ax_full, ax_zoom = axes[i]

        # Left panel: full distribution
        ax_full.hist(gates, bins=100, color="steelblue", edgecolor="none")
        ax_full.set_xlim(0, 1)
        ax_full.set_xlabel("Gate Value (sigmoid output)")
        ax_full.set_ylabel("Count")
        ax_full.set_title(
            f"λ = {lam}  |  Accuracy = {acc:.1%}  |  Sparsity = {spar:.1%}",
            fontsize=11
        )
        ax_full.axvline(0.01, color="red", linestyle="--", linewidth=1,
                        label="prune threshold (0.01)")
        ax_full.legend(fontsize=9)

        
        near_zero = gates[gates < 0.1]
        ax_zoom.hist(near_zero, bins=50, color="salmon", edgecolor="none")
        ax_zoom.set_xlabel("Gate Value — zoom [0, 0.1]")
        ax_zoom.set_ylabel("Count")
        ax_zoom.set_title(f"λ = {lam}  |  Near-zero (pruned) gates")

    fig.suptitle(
        "Soft Gate Distributions — Self-Pruning Network (CIFAR-10)",
        fontsize=13, y=1.01
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\nPlot saved → '{save_path}'")



# Main


def main():

    lambdas = [0.1, 1.0, 5.0]
    EPOCHS  = 30

    results = {}
    for lam in lambdas:
        acc, spar, gates = train(lam=lam, epochs=EPOCHS)
        results[lam] = (acc, spar, gates)


    print("\n\n" + "="*55)
    print("  RESULTS SUMMARY")
    print("="*55)
    print(f"  {'Lambda':<10} {'Test Accuracy':>15} {'Sparsity Level':>16}")
    print("  " + "-"*41)
    for lam in sorted(results):
        acc, spar, _ = results[lam]
        print(f"  {lam:<10.1f} {acc:>14.2%} {spar:>15.2%}")
    print("="*55)


    plot_gate_distributions(results)


if __name__ == "__main__":
    main()

Using device: cuda

  Training with λ = 0.1
  Epoch   1/30 | Total loss: 1.8236 | CLS loss: 1.7737 | Test acc: 0.442 | Sparsity: 0.00%
  Epoch   5/30 | Total loss: 1.4635 | CLS loss: 1.4187 | Test acc: 0.525 | Sparsity: 0.00%
  Epoch  10/30 | Total loss: 1.3401 | CLS loss: 1.2997 | Test acc: 0.557 | Sparsity: 0.00%
  Epoch  15/30 | Total loss: 1.2508 | CLS loss: 1.2130 | Test acc: 0.582 | Sparsity: 0.00%
  Epoch  20/30 | Total loss: 1.1778 | CLS loss: 1.1411 | Test acc: 0.601 | Sparsity: 0.00%
  Epoch  25/30 | Total loss: 1.1257 | CLS loss: 1.0894 | Test acc: 0.611 | Sparsity: 0.01%
  Epoch  30/30 | Total loss: 1.0986 | CLS loss: 1.0623 | Test acc: 0.614 | Sparsity: 0.01%

  ✓ Final  →  Accuracy: 0.614  |  Sparsity: 0.01%

  Training with λ = 1.0
  Epoch   1/30 | Total loss: 2.2264 | CLS loss: 1.7771 | Test acc: 0.427 | Sparsity: 0.00%
  Epoch   5/30 | Total loss: 1.6788 | CLS loss: 1.4166 | Test acc: 0.525 | Sparsity: 0.03%
  Epoch  10/30 | Total loss: 1.4816 | CLS loss: 1.2943 | Test